# Connection catalog notebook

This notebook shows how to load **named SQL** and **named filesystem** profiles into `ConnectionCatalog`, then use them through `SqlDatabaseResource` and `ParquetDataResource`.

In [1]:
import sys
import tempfile
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src" / "boti").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import pandas as pd
from sqlalchemy import create_engine, text

from boti_data import ConnectionCatalog, ParquetDataConfig, ParquetDataResource

print(PROJECT_ROOT)

/Users/lvalverdeb/TeamDev/repo-split/boti-data


## Create a temporary environment and seed data

In [2]:
tmp_root = Path(tempfile.mkdtemp(prefix="boti_catalog_"))
sql_db = tmp_root / "primary.db"
parquet_root = tmp_root / "warehouse"
parquet_root.mkdir(parents=True)
env_file = tmp_root / ".env.catalog"

items = pd.DataFrame(
    {
        "id": [1, 2, 3],
        "name": ["Alice", "Bob", "Carla"],
        "team": ["analytics", "platform", "analytics"],
    }
)

engine = create_engine(f"sqlite:///{sql_db}")
with engine.begin() as conn:
    items.to_sql("people", conn, index=False, if_exists="replace")
items.to_parquet(parquet_root / "people.parquet", index=False)
engine.dispose()

env_file.write_text(
    "\n".join(
        [
            f"PRIMARY_DB_CONNECTION_URL=sqlite:///{sql_db}",
            "PRIMARY_DB_QUERY_ONLY=false",
            "WAREHOUSE_FS_TYPE=file",
            f"WAREHOUSE_FS_PATH={parquet_root}",
        ]
    )
    + "\n",
    encoding="utf-8",
)

env_file.read_text()

'PRIMARY_DB_CONNECTION_URL=sqlite:////var/folders/j1/c0fy94996q51rf0nkcvg577m0000gn/T/boti_catalog_4ww18tz3/primary.db\nPRIMARY_DB_QUERY_ONLY=false\nWAREHOUSE_FS_TYPE=file\nWAREHOUSE_FS_PATH=/var/folders/j1/c0fy94996q51rf0nkcvg577m0000gn/T/boti_catalog_4ww18tz3/warehouse\n'

## Load named profiles into the catalog

In [3]:
catalog = ConnectionCatalog()
catalog.load_sql("primary", "PRIMARY_DB_", env_file=env_file)
catalog.load_filesystem("warehouse", "WAREHOUSE_", env_file=env_file)

primary_config = catalog.sql_config("primary")
warehouse_config = catalog.filesystem_config("warehouse")

print(primary_config)
print(warehouse_config)

verbose=False debug=False logger=None allow_pickle=False project_root=None extra_allowed_paths=[] connection_url=SecretStr('**********') query_only=False worker_connection_env_var=None pool_size=5 max_overflow=10 pool_timeout=30 pool_recycle=1800 pool_pre_ping=True poolclass=<class 'sqlalchemy.pool.impl.QueuePool'> connect_args={} execution_options={}
fs_type='file' fs_path='/var/folders/j1/c0fy94996q51rf0nkcvg577m0000gn/T/boti_catalog_4ww18tz3/warehouse' fs_key=None fs_secret=None fs_endpoint=None fs_token=None fs_region=None fs_verify_ssl=True fs_connect_timeout=10.0 fs_read_timeout=30.0 fs_options={}


## Use the named SQL profile

In [4]:
with catalog.create_sql_resource("primary") as db:
    with db.session() as session:
        rows = session.execute(text("SELECT id, name, team FROM people ORDER BY id")).mappings().all()

rows

[{'id': 1, 'name': 'Alice', 'team': 'analytics'},
 {'id': 2, 'name': 'Bob', 'team': 'platform'},
 {'id': 3, 'name': 'Carla', 'team': 'analytics'}]

## Use the named filesystem profile through the parquet resource

In [5]:
parquet_config = ParquetDataConfig(
    project_root=tmp_root,
    filesystem_profile="warehouse",
    parquet_filename="people",
)

with ParquetDataResource(parquet_config, catalog=catalog) as resource:
    parquet_df = resource.load_filtered({"team__exact": "analytics"}).compute().sort_values("id")

parquet_df.reset_index(drop=True)

,id,name,team
0,1,Alice,analytics
1,3,Carla,analytics


## Inspect the underlying filesystem adapters

In [6]:
fs = catalog.filesystem("warehouse")
arrow_fs, arrow_path = catalog.pyarrow_filesystem("warehouse")

print(type(fs).__name__)
print(type(arrow_fs).__name__)
print(arrow_path)

LocalFileSystem
LocalFileSystem
/var/folders/j1/c0fy94996q51rf0nkcvg577m0000gn/T/boti_catalog_4ww18tz3/warehouse
